In [1]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split

I0000 00:00:1779546563.425485    5467 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
IMG_SIZE = 256  # Standardized input resolution for both phases
BATCH_SIZE = 4 # Batch size
EPOCHS_PHASE1 = 100  # Semantic Segmentation Focus, for Segmentation 
EPOCHS_PHASE2 = 30   # Segmentation-Based Classification Focus for classification Polyp and Non-Polyp

#### Declaring dataset path for segmentation and classification

In [3]:
# Directory and Artifact Paths
DATA_PATH = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
POSITIVE_DIR = os.path.join(DATA_PATH, 'imagesAll_positive') # For Polyp classification
NEGATIVE_DIR = os.path.join(DATA_PATH, 'sequenceData/negativeOnly')  # For non polyp classification
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/unified_net/thesis_v3_sequential.keras"

#### mixed_precision
Activate mixed precision training for efficiency and memory optimization
It is a performance optimization for GPU. It utilize half of the GPU Memory, 
increases the the model training speed 3 times without effecting on accuracy. 
mixed_float16, 16 bits helps to get speedy calculation when need, Flawless accuracy performed by 32-bits

In [4]:

mixed_precision.set_global_policy('mixed_float16')

#### Image Augmentation
As we have limited datasources we have to augment images. If we train with limited datasets Model can memorize 
instead of learning which causes overfitting. An overfitted model cannot predict other image except training dataset. Even it cannot predict on validation dataset as well.
The overfitting occurs when The Training Loss going down b

In [ ]:
#
def augment(img, mask):
    """Applies shape-preserving geometric and radiometric transformations."""
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask